Train meta learner bằng tập valid, sau đó test trên tập test.

In [1]:
import pandas as pd
data_valid = pd.read_parquet(r"final_data\chunk-based-split-3\valid_df_prepared.parquet")
data_test = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet")

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
import pandas as pd
import numpy as np
import copy
import gc
import os

# mô hình GatEmbedder
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=4):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # lớp Linear để tính điểm số (score) cho từng time-step
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # đầu ra bi-lstm: (Batch, SeqLen, Hidden*2)
        
        # tính điểm chú tý cho mỗi timestap => (batch, seqlen, 1)
        scores = self.attention(lstm_outputs)
        
        # áp dung softmax để đưa sequence attention scores về dạng xác suất
        weights = torch.softmax(scores, dim=1)
        
        # nhân trọng số với output của lstm và tỉnh tổng (nhân chập)
        # (Batch, SeqLen, 1) * (Batch, SeqLen, Hidden*2) -> (Batch, Hidden*2)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        
        # trả lại context_vector và weights (để phục vụ cho việc trực quan hóa attention sau này)
        return context_vector, weights
    
# mô hình CNN-BiLSTM-Attention
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, padding=1)
        # THAY BatchNorm BẰNG GroupNorm ĐỂ CHỐNG HIỆU ỨNG TRÔI GIÁ TRỊ TOÀN CỤC Ở SPLIT-3
        self.bn1 = nn.GroupNorm(num_groups=8, num_channels=64) 
        self.dropout_cnn = nn.Dropout1d(0.2)

        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi, tăng kênh lên 128
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.GroupNorm(num_groups=16, num_channels=128)
        
        self.relu = nn.ReLU()
        
        # giảm 2 timesteps lại còn 1
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # bi-lstm như cũ
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        

        # kích thước đầu vào của Attention = hidden_size * 2 (vì BiLSTM ghép 2 chiều ngược xuôi)
        self.attention = Attention(hidden_size * 2)
        
        self.dropout = nn.Dropout(0.5)
        
        # tầng phân loại cuối cùng
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # đầu vào (256, 10, 314)
        
        # đảo trục cho CNN -> (256, 314, 10)
        x = x.permute(0, 2, 1) 
        
        # đi qua cnn block 1 => (256, 64, 10)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout_cnn(x)

        # đi qua cnn block 2 => (256, 128, 10)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        # giảm chiều dài chuỗi đi 2 lần => (256, 128, 5)
        x = self.pool(x)
        
        # đảo trục lại cho lstm (256, 5, 128)
        x = x.permute(0, 2, 1)
        
        # đi qua bi-lstm => (256, 5, 256)
        out, (hn, cn) = self.bilstm(x)
        
        # dùng attention để tính ra context vector: (256, 256)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua lớp dense để ra dự đoán nhãn
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


# tạo đồ thị GAT cho tập valid và test (cần tách riêng để mô phỏng thực tế (không gộp lại giống lúc train))
def build_inference_graph(df, window_size=10):
    print("Pre-processing Timestamp cho Inference Graph...")
    if 'timestamp_num' not in df.columns:
        df['timestamp_num'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
        
    df.sort_values(by='timestamp_num', inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    cols_present = [c for c in cols_to_drop if c in df.columns]
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
    features_np = np.ascontiguousarray(features_np)
    
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()

    timestamps = df['timestamp_num'].values
    src_ips = df['src_ip'].values
    dst_ips = df['dst_ip'].values
    src_ports = df['src_port'].astype(str).values
    dst_ports = df['dst_port'].astype(str).values
    
    ip_to_indices = {}
    from tqdm.auto import tqdm
    for idx in tqdm(range(len(df)), desc="[1/2] Build IP Dictionary"):
        s_ip, d_ip = src_ips[idx], dst_ips[idx]
        if s_ip not in ip_to_indices: ip_to_indices[s_ip] = []
        if len(ip_to_indices[s_ip]) == 0 or ip_to_indices[s_ip][-1] != idx: ip_to_indices[s_ip].append(idx)
        if d_ip not in ip_to_indices: ip_to_indices[d_ip] = []
        if len(ip_to_indices[d_ip]) == 0 or ip_to_indices[d_ip][-1] != idx: ip_to_indices[d_ip].append(idx)
            
    all_src, all_dst, all_edge_attrs = [], [], []
    for ip, indices in tqdm(ip_to_indices.items(), desc=f"[2/2] Nối lưới đồ thị (1-chiều Inference)"):
        n_flows = len(indices)
        if n_flows < 2: continue
        for i in range(1, n_flows):
            curr_idx = indices[i]
            for j in range(max(0, i - window_size), i):
                past_idx = indices[j]
                dt_raw = abs(timestamps[curr_idx] - timestamps[past_idx])
                if dt_raw > 60.0: continue
                dt = np.log1p(dt_raw * 1e6) / 18.0
                same_dir = 1.0 if src_ips[curr_idx] == src_ips[past_idx] else 0.0
                same_sport = 1.0 if src_ports[curr_idx] == src_ports[past_idx] else 0.0
                same_dport = 1.0 if dst_ports[curr_idx] == dst_ports[past_idx] else 0.0
                attr = [dt, same_dir, same_sport, same_dport]
                
                all_src.append(past_idx)
                all_dst.append(curr_idx)
                all_edge_attrs.append(attr)
                
    df_edges = pd.DataFrame({'src': all_src, 'dst': all_dst, 'a0': [a[0] for a in all_edge_attrs], 'a1': [a[1] for a in all_edge_attrs], 'a2': [a[2] for a in all_edge_attrs], 'a3': [a[3] for a in all_edge_attrs]})
    df_edges.drop_duplicates(subset=['src', 'dst'], inplace=True)
    df_edges.reset_index(drop=True, inplace=True)
    
    edge_index = torch.tensor(df_edges[['src', 'dst']].values.T, dtype=torch.long).contiguous()
    edge_attr = torch.tensor(df_edges[['a0', 'a1', 'a2', 'a3']].values, dtype=torch.float32).contiguous()
    
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    graph.mask = torch.ones(len(df), dtype=torch.bool) 
    
    del features_np, df_edges, all_src, all_dst, all_edge_attrs
    gc.collect()
    return graph, df

# trích xuất embeddings GAT thực tế bằng NeighborLoader
@torch.no_grad()
def extract_embeddings_mask(model, graph_data, device='cpu'):
    model.eval()
    # Tăng mức lấy mẫu hàng xóm lên mức rất cao [100, 30] để bao quát được toàn bộ mạng lưới Botnet/DDoS lớn
    loader = NeighborLoader(graph_data, num_neighbors=[200, 50], input_nodes=graph_data.mask, batch_size=512, shuffle=False, num_workers=0)
    all_embeddings = []
    from tqdm.auto import tqdm
    pbar = tqdm(loader, desc="Extracting GAT Embeddings")
    for batch in pbar:
        batch = batch.to(device)
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        all_embeddings.append(emb[:batch.batch_size].cpu().numpy())
    pbar.close()
    return np.concatenate(all_embeddings, axis=0)

# trích xuất xác suất dự đoán từ mô hình CNN-BiLSTM-Attention
@torch.no_grad()
def extract_probs_cnn_bilstm(model, X_windows, batch_size=512, device='cpu'):
    model.eval()
    dataset = torch.utils.data.TensorDataset(torch.tensor(X_windows, dtype=torch.float32))
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    all_probs = []
    from tqdm.auto import tqdm
    pbar = tqdm(loader, desc="Extracting CNN-BiLSTM Probs")
    for batch in pbar:
        inputs = batch[0].to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()  
        all_probs.append(probs)
    pbar.close()
    return np.concatenate(all_probs, axis=0)

In [3]:
import numpy as np
from tqdm.auto import tqdm
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

def get_window_data_with_index(df, window_size=10):
    """
    trượt cửa sổ bằng chỉ số mảng (numpy array index) đẻ nối output của model (GAT_XGB)
    """
    ignore_cols = ['label', 'flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']
    feature_cols = [c for c in df.columns if c not in ignore_cols]
    
    features = df[feature_cols].values.astype(np.float32)
    labels = df['label'].values
    
    X_windows = []
    y_targets = []
    target_indices = []
    
    print(f"Processing sliding windows (size={window_size})...")
    for i in tqdm(range(len(df) - window_size + 1)):
        window = features[i : i + window_size]
        target_idx = i + window_size - 1
        target_label = labels[target_idx]
        
        X_windows.append(window)
        y_targets.append(target_label)
        target_indices.append(target_idx)
        
    return np.array(X_windows), np.array(y_targets), np.array(target_indices)

# xây dựng đồ thị và trích xuất embeddings GAT cho tập valid và test
device = 'cuda' if torch.cuda.is_available() else 'cpu'

num_features = len([c for c in data_valid.columns if c not in ['label', 'flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']])
num_classes = len(data_valid['label'].unique())

# load gat embedder
print(f"Đang Load trọng số GAT_Embedder (features = {num_features}, classes = {num_classes})...")
gat_model = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=num_classes, heads=8, edge_dim=4).to(device)
gat_model.load_state_dict(torch.load(r"model_final\gat_embedder.pth", map_location=device))

# xây dựng inference graph (tách riêng cho valid và test để mô phỏng thực tế, không gộp lại giống lúc train)
print("\n--- XÂY DỰNG INFERENCE GRAPH (Mất một chút RAM và Thời gian) ---")
graph_valid, data_valid = build_inference_graph(data_valid)
graph_test, data_test = build_inference_graph(data_test)

# trích xuất embeddings từ valid và test
print("\n--- TRÍCH XUẤT EMBEDDINGS TỪ VALID VÀ TEST ---")
valid_gat_emb = extract_embeddings_mask(gat_model, graph_valid, device=device)
test_gat_emb  = extract_embeddings_mask(gat_model, graph_test, device=device)

# tạo sliding windows cho CNN-BiLSTM (không có under-sampling, giữ nguyên thứ tự thời gian)
X_val_win, y_val_win, val_align_idx = get_window_data_with_index(data_valid, window_size=10)
X_test_win, y_test_win, test_align_idx = get_window_data_with_index(data_test, window_size=10)

print(f"\n[Valid] Windows Shape: {X_val_win.shape} | Align Indices: {val_align_idx.shape}")
print(f"[Test]  Windows Shape: {X_test_win.shape}  | Align Indices: {test_align_idx.shape}")

# sử dụng mô hình XGBoost đã huấn luyện để trích xuất xác suất dự đoán (probabilities) cho tập valid và test
print("\n--- Load XGBoost Model và Trích xuất xác suất (Probabilities) ---")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(r"model_final\GAT_XGB_Hybrid_FAISS_Model.json")

prob_valid_xgb = xgb_model.predict_proba(valid_gat_emb)
prob_test_xgb  = xgb_model.predict_proba(test_gat_emb)

print("\n--- Load CNN-BiLSTM_Attention Model và Trích xuất xác suất ---")
cnn_model = CNN_BiLSTM_Attention(num_features=num_features, num_classes=num_classes, time_steps=10, hidden_size=128).to(device)
cnn_model.load_state_dict(torch.load(r"model_final\best_cnn_bilstm.pth", map_location=device))

# extract probabilities từ CNN-BiLSTM-Attention cho tập valid và test
prob_valid_bilstm = extract_probs_cnn_bilstm(cnn_model, X_val_win, batch_size=512, device=device)
prob_test_bilstm  = extract_probs_cnn_bilstm(cnn_model, X_test_win, batch_size=512, device=device)


# rút ra các dự đoán XGBoost (Node level) tương ứng với phần tử cuối cùng của (Sequence Level window)
aligned_prob_valid_xgb = prob_valid_xgb[val_align_idx]
aligned_prob_test_xgb  = prob_test_xgb[test_align_idx]

# nối đặc trưng meta: XGB Probabilities + CNN-BiLSTM Probabilities
X_meta_valid = np.concatenate([aligned_prob_valid_xgb, prob_valid_bilstm], axis=1)
y_meta_valid = y_val_win 

X_meta_test = np.concatenate([aligned_prob_test_xgb, prob_test_bilstm], axis=1)
y_meta_test = y_test_win

print(f"\n[Meta-Dataset] Validation Shape: {X_meta_valid.shape}")
print(f"[Meta-Dataset] Test Shape:       {X_meta_test.shape}")


# meta-learner dùng XGBoost
print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost ---")

meta_model = xgb.XGBClassifier(
    n_estimators=300,      
    max_depth=3,           # =đặt độ sâu nông
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42
)

# huấn luyện trên tập meta (tập test không tham gia bất cứ gì về huấn luyện)
meta_model.fit(
    X_meta_valid, y_meta_valid,
    eval_set=[(X_meta_test, y_meta_test)],
    verbose=False
)
final_preds = meta_model.predict(X_meta_test)

print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE TRÊN TẬP TEST ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, final_preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, final_preds, digits=4))

Đang Load trọng số GAT_Embedder (features = 314, classes = 11)...

--- XÂY DỰNG INFERENCE GRAPH (Mất một chút RAM và Thời gian) ---
Pre-processing Timestamp cho Inference Graph...


C:\Users\Admin\AppData\Local\Temp\ipykernel_20628\87762446.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gat_model.load_state_dict(torch.load(r"model_final\gat_embedd

[1/2] Build IP Dictionary:   0%|          | 0/570134 [00:00<?, ?it/s]

[2/2] Nối lưới đồ thị (1-chiều Inference):   0%|          | 0/18 [00:00<?, ?it/s]

Pre-processing Timestamp cho Inference Graph...


...


[Valid] Windows Shape: (570125, 10, 314) | Align Indices: (570125,)
[Test]  Windows Shape: (760231, 10, 314)  | Align Indices: (760231,)

--- Load XGBoost Model và Trích xuất xác suất (Probabilities) ---

--- Load CNN-BiLSTM_Attention Model và Trích xuất xác suất ---


C:\Users\Admin\AppData\Local\Temp\ipykernel_20628\87762446.py:71: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(r"model_final\best_cnn_b

Extracting CNN-BiLSTM Probs:   0%|          | 0/1114 [00:00<?, ?it/s]

Extracting CNN-BiLSTM Probs:   0%|          | 0/1485 [00:00<?, ?it/s]


[Meta-Dataset] Validation Shape: (570125, 22)
[Meta-Dataset] Test Shape:       (760231, 22)

--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost ---

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE TRÊN TẬP TEST ---
Accuracy:         99.22%
F1-Score (Macro): 96.62%

Classification Report:

              precision    recall  f1-score   support

           0     0.9862    0.9939    0.9901     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.9339    1.0000    0.9658      2515
           3     1.0000    0.9889    0.9944     36253
           4     0.9070    0.9062    0.9066     18979
           5     1.0000    1.0000    1.0000      2451
           6     0.9816    0.7576    0.8552     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8965    0.9978    0.9444     16746
           9     0.9618    0.9829    0.9722     41514
          10     1.0000    0.9995    0.9997     18567

    accuracy                         0.9922    760231
   

Meta Learner bằng Logistic Regression

In [4]:
print("\n--- BẮT ĐẦU TUNING META-LEARNER (LOGISTIC REGRESSION) ---")
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Định nghĩa tập các tham số cần tune
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],               # Độ mạnh của Regularization (ngược)
    'solver': ['lbfgs', 'liblinear'],           # Thuật toán tối ưu
    'class_weight': [None, 'balanced']          # Xử lý mất cân bằng nhãn
}

# Khởi tạo mô hình cơ sở
lr_base = LogisticRegression(max_iter=2000, random_state=42)

# Sử dụng GridSearchCV với 5-Fold Cross Validation trên tập Valid
grid_search = GridSearchCV(
    estimator=lr_base,
    param_grid=param_grid,
    scoring='f1_macro',                         # Focus vào Macro F1-Score
    cv=5,
    n_jobs=-1,
    verbose=1
)

print("Đang thực hiện tìm kiếm siêu tham số tối ưu trên tập valid...")
grid_search.fit(X_meta_valid, y_meta_valid)

print(f"\n[+] Param tốt nhất tìm được: {grid_search.best_params_}")
print(f"[+] Validation F1 (Cross-Val): {grid_search.best_score_ * 100:.2f}%")

# Sử dụng mô hình tốt nhất từ GridSearch cho tập test thực tế
best_lr_model = grid_search.best_estimator_
lr_final_preds = best_lr_model.predict(X_meta_test)

print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE (TUNED LOGISTIC REGRESSION) ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, lr_final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, lr_final_preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, lr_final_preds, digits=4))


--- BẮT ĐẦU TUNING META-LEARNER (LOGISTIC REGRESSION) ---
Đang thực hiện tìm kiếm siêu tham số tối ưu trên tập valid...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

[+] Param tốt nhất tìm được: {'C': 10, 'class_weight': None, 'solver': 'liblinear'}
[+] Validation F1 (Cross-Val): 99.22%

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE (TUNED LOGISTIC REGRESSION) ---
Accuracy:         98.89%
F1-Score (Macro): 95.87%

Classification Report:

              precision    recall  f1-score   support

           0     0.9662    0.9835    0.9748     19846
           1     1.0000    1.0000    1.0000    484077
           2     1.0000    0.9996    0.9998      2515
           3     1.0000    0.9866    0.9933     36253
           4     0.8502    0.8754    0.8626     18979
           5     1.0000    1.0000    1.0000      2451
           6     0.9864    0.7198    0.8323     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8842    0.9982    0.9378    

Thêm đặc trưng tương tác (Interaction Term): [p1, p2, p1 * p2] cho Logistic Regression

In [5]:
print("\n--- BẮT ĐẦU TUNING META-LEARNER VỚI ĐẶC TRƯNG TƯƠNG TÁC (INTERACTION TERMS) ---")
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Tạo đặc trưng tương tác: p1 * p2
interaction_valid = aligned_prob_valid_xgb * prob_valid_bilstm
interaction_test  = aligned_prob_test_xgb  * prob_test_bilstm

# Nối các đặc trưng: [p1, p2, p1 * p2]
X_meta_valid_inter = np.concatenate([aligned_prob_valid_xgb, prob_valid_bilstm, interaction_valid], axis=1)
X_meta_test_inter  = np.concatenate([aligned_prob_test_xgb, prob_test_bilstm, interaction_test], axis=1)

print(f"[Meta-Dataset with Interaction] Validation Shape: {X_meta_valid_inter.shape}")
print(f"[Meta-Dataset with Interaction] Test Shape:       {X_meta_test_inter.shape}")

# Sử dụng trực tiếp cấu hình siêu tham số tốt nhất
best_lr_inter_model = LogisticRegression(
    C=100, 
    class_weight=None, 
    solver='liblinear', 
    max_iter=2000, 
    random_state=42
)

print("Đang huấn luyện Logistic Regression (Interaction) với bộ param tốt nhất...")
best_lr_inter_model.fit(X_meta_valid_inter, y_meta_valid)

# Sử dụng mô hình tốt nhất cho tập test thực tế
lr_inter_final_preds = best_lr_inter_model.predict(X_meta_test_inter)

print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE (LOGISTIC REGRESSION + INTERACTION) ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, lr_inter_final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, lr_inter_final_preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, lr_inter_final_preds, digits=4))


--- BẮT ĐẦU TUNING META-LEARNER VỚI ĐẶC TRƯNG TƯƠNG TÁC (INTERACTION TERMS) ---
[Meta-Dataset with Interaction] Validation Shape: (570125, 33)
[Meta-Dataset with Interaction] Test Shape:       (760231, 33)
Đang huấn luyện Logistic Regression (Interaction) với bộ param tốt nhất...

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE (LOGISTIC REGRESSION + INTERACTION) ---
Accuracy:         98.95%
F1-Score (Macro): 96.05%

Classification Report:

              precision    recall  f1-score   support

           0     0.9709    0.9829    0.9768     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.9988    0.9996    0.9992      2515
           3     1.0000    0.9876    0.9937     36253
           4     0.8518    0.8965    0.8736     18979
           5     1.0000    1.0000    1.0000      2451
           6     0.9810    0.7231    0.8325     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8936    0.9942    0.9412     16746
         

In [6]:
# train XGBoost meta-learner với đặc trưng tương tác
print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost VỚI ĐẶC TRƯNG TƯƠNG TÁC ---")
meta_model_inter = xgb.XGBClassifier(
    n_estimators=300,      
    max_depth=3,           
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42
)
meta_model_inter.fit(X_meta_valid_inter, y_meta_valid, eval_set=[(X_meta_test_inter, y_meta_test)], verbose=False)
final_preds_inter = meta_model_inter.predict(X_meta_test_inter)
print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE (XGBOOST META-LEARNER + INTERACTION) ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, final_preds_inter)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, final_preds_inter, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, final_preds_inter, digits=4))



--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost VỚI ĐẶC TRƯNG TƯƠNG TÁC ---

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE (XGBOOST META-LEARNER + INTERACTION) ---
Accuracy:         99.20%
F1-Score (Macro): 96.80%

Classification Report:

              precision    recall  f1-score   support

           0     0.9808    0.9949    0.9878     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.9809    1.0000    0.9904      2515
           3     1.0000    0.9862    0.9931     36253
           4     0.9088    0.9061    0.9074     18979
           5     1.0000    1.0000    1.0000      2451
           6     0.9817    0.7569    0.8548     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8978    0.9980    0.9453     16746
           9     0.9573    0.9821    0.9696     41514
          10     1.0000    0.9996    0.9998     18567

    accuracy                         0.9920    760231
   macro avg     0.9734    0.9658    0.9680    76